# Metabolic pathway maps

In [1]:
# Setup. On Google Colab this downloads the course files and installs the dependencies (~2 min).
# Locally (conda env from environment.yml) it only moves to the course folder so `data/` is found.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.isdir("/content/cell-factory-design-course"):
        !git clone -q --depth 1 https://github.com/lab-biotek-bio-ugm/cell-factory-design-course /content/cell-factory-design-course
        %pip install -q --progress-bar off -r /content/cell-factory-design-course/requirements.txt
    from google.colab import output
    output.enable_custom_widget_manager()  # needed for Escher maps

for folder in (".", "..", "/content/cell-factory-design-course"):
    if os.path.isdir(os.path.join(folder, "data")):
        os.chdir(folder)
        break

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
Reason for being yanked: deprecated


## Overview

**Learning objectives.** After this notebook you can:
1. Load a published Escher map and list the available maps.
2. Overlay reaction data (e.g. FBA fluxes) and metabolite data on a map.
3. Read an Escher map: arrow direction, color and width scales.

**Estimated time:** 30 min (10 min walkthrough, 15 min exercise).

**Prerequisites:** notebooks 01–02.

### Background

A flux vector from FBA has thousands of numbers, which is hard to inspect in a table. **Escher** ([King et al., 2015](https://doi.org/10.1371/journal.pcbi.1004321)) draws metabolic pathway maps in the browser: metabolites are circles (nodes), reactions are arrows (edges). Maps are drawn by hand for a specific model namespace (here BiGG identifiers), so reaction data given as `{reaction_id: value}` is matched to the arrows by ID. Reactions in the data that are not on the map are simply not shown.

When you pass `reaction_data`, Escher colors and scales the arrows by value; arrowheads follow the sign of the flux (a negative flux runs "backwards" relative to how the reaction is written). `metabolite_data` works the same way for the nodes (e.g. metabolomics concentrations). You can pan, zoom and hover over elements in the widget to see their IDs and values.

<div class="alert alert-info">

**Note:** Escher maps are interactive widgets. They need a running kernel (they do not show in a static preview of the notebook), and the maps are downloaded from the Escher website the first time you use them, so you need an internet connection.

</div>

## Metabolic pathway visualizations with Escher

In [2]:
import escher

Display all available maps in Escher. Map names start with the model they were drawn for (e.g. `e_coli_core.`, `iJO1366.`).

In [3]:
escher.list_available_maps()

[{'organism': 'Saccharomyces cerevisiae',
  'map_name': 'iMM904.Central carbon metabolism'},
 {'organism': 'Homo sapiens',
  'map_name': 'RECON1.Inositol retinol metabolism'},
 {'organism': 'Homo sapiens', 'map_name': 'RECON1.Glycolysis TCA PPP'},
 {'organism': 'Homo sapiens', 'map_name': 'RECON1.Tryptophan metabolism'},
 {'organism': 'Homo sapiens', 'map_name': 'RECON1.Carbohydrate metabolism'},
 {'organism': 'Homo sapiens',
  'map_name': 'RECON1.Amino acid metabolism (partial)'},
 {'organism': 'Escherichia coli', 'map_name': 'iJO1366.Nucleotide metabolism'},
 {'organism': 'Escherichia coli',
  'map_name': 'iJO1366.Fatty acid biosynthesis (saturated)'},
 {'organism': 'Escherichia coli',
  'map_name': 'iJO1366.Nucleotide and histidine biosynthesis'},
 {'organism': 'Escherichia coli', 'map_name': 'e_coli_core.Core metabolism'},
 {'organism': 'Escherichia coli', 'map_name': 'iJO1366.Central metabolism'},
 {'organism': 'Escherichia coli',
  'map_name': 'iJO1366.Fatty acid beta-oxidation'}

Display a map of _E. coli_ central carbon metabolism.

In [4]:
escher.Builder('e_coli_core.Core metabolism')

Visualize reaction-centric and/or metabolite-centric data. Here we give phosphoglycerate kinase (PGK) a value of 100 and ATP a value of 20, so only those two elements get colored.

In [5]:
escher.Builder('e_coli_core.Core metabolism',
               reaction_data={'PGK': 100},
               metabolite_data={'ATP': 20})

In [14]:
escher.Builder('e_coli_core.Core metabolism',
               reaction_data={'GLUDy': 100},
               metabolite_data={'ATP': 50})

## Exercise (15 min)

* Solve the _E. coli_ from the previous session and visualize the computed fluxes on the central carbon metabolism map.
* Visualize the same flux distribution on another map (a list of available maps can be retrieved by running `escher.list_available_maps()`)

### Solution to the exercise

In [10]:
import cobra
import os

# Path to the E. coli core model file (assuming it's in the data folder of the course materials)
# The setup cell changes the working directory, so 'data/e_coli_core.xml.gz' should be accessible.
model_path = os.path.join("data", "e_coli_core.xml.gz")

# Load the E. coli core model from the local file
try:
    model = cobra.io.read_sbml_model(model_path)
    print(f"Successfully loaded model from local path: {model_path}")
except FileNotFoundError:
    print(f"Error: Model file not found at {model_path}. Trying web loading as fallback...")
    # Fallback to web loading if local file not found, though it previously failed.
    model = cobra.io.load_model('e_coli_core')

# Perform Flux Balance Analysis (FBA)
# By default, FBA maximizes biomass production
solution = model.optimize()

# Extract reaction fluxes
fluxes = solution.fluxes.to_dict()

print("FBA solved successfully. Displaying the first 10 fluxes:")
for i, (rxn_id, flux_val) in enumerate(list(fluxes.items())[:10]):
    print(f"{rxn_id}: {flux_val:.4f}")

Successfully loaded model from local path: data/e_coli_core.xml.gz
FBA solved successfully. Displaying the first 10 fluxes:
ACALD: 0.0000
ACALDt: 0.0000
ACKr: 0.0000
ACONTa: 6.0072
ACONTb: 6.0072
ACt2r: 0.0000
ADK1: 0.0000
AKGDH: 5.0644
AKGt2r: 0.0000
ALCD2x: 0.0000


#### Visualize fluxes on the `e_coli_core.Core metabolism` map

In [11]:
escher.Builder('e_coli_core.Core metabolism',
               reaction_data=fluxes)

#### Visualize the same fluxes on the `iJO1366.Central metabolism` map

In [12]:
escher.Builder('iJO1366.Central metabolism',
               reaction_data=fluxes)

## References

- King, Dräger, Ebrahim et al. (2015). Escher: A Web Application for Building, Sharing, and Embedding Data-Rich Visualizations of Biological Pathways. *PLOS Computational Biology* 11:e1004321. [doi.org/10.1371/journal.pcbi.1004321](https://doi.org/10.1371/journal.pcbi.1004321)